In [ ]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

In [ ]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [ ]:
# Keep only books with at least 100 ratings.
book_rating_counts = df_ratings.groupby('isbn')['rating'].count()
valid_isbns = book_rating_counts[book_rating_counts >= 100].index
good_books = df_books[df_books['isbn'].isin(valid_isbns)].copy()

# Keep only users with at least 200 ratings.
user_rating_counts = df_ratings.groupby('user')['rating'].count()
valid_users = user_rating_counts[user_rating_counts >= 200].index

# Apply both filters to the ratings dataset.
filtered_ratings = df_ratings[df_ratings['user'].isin(valid_users)]
filtered_ratings = filtered_ratings[filtered_ratings['isbn'].isin(good_books['isbn'])]

# Build the book-user feature matrix and convert it to a sparse matrix.
book_user_matrix = filtered_ratings.pivot(
    index='isbn',
    columns='user',
    values='rating'
).fillna(0)

book_user_sparse = csr_matrix(book_user_matrix.values)

# Fit KNN using cosine distance, matching the challenge's expected results.
model = NearestNeighbors(metric='cosine')
model.fit(book_user_sparse)

In [ ]:
# function to return recommended books - this will be tested
def get_recommends(book = ""):
  matching_books = good_books.loc[good_books['title'] == book]

  if matching_books.empty:
    return [book, []]

  book_rows = book_user_matrix.loc[
      book_user_matrix.index.isin(matching_books['isbn'])
  ]

  distances, indices = model.kneighbors(
      [row for row in book_rows.values],
      n_neighbors=6
  )

  distances = distances[0][1:]
  indices = indices[0][1:]

  titles = [
      df_books.loc[
          df_books['isbn'] == book_user_matrix.iloc[i].name,
          'title'
      ].values[0]
      for i in indices
  ]

  recommendations = [
      [title, float(distance)]
      for title, distance in zip(titles, distances)
  ][::-1]

  return [book, recommendations]

In [ ]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2): 
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()